# Superstore Sales Forecasting Analysis\n
\n
EDA + ML (RF, ARIMA) + forecasts. Generates all files for dashboard.

In [ ]:
import sys, os, pd as pd, np as np, matplotlib.pyplot as plt, seaborn as sns\n
from sklearn.ensemble import RandomForestRegressor; from sklearn.metrics import mean_absolute_error, mean_squared_error\n
from statsmodels.tsa.arima.model import ARIMA; import joblib\n
sys.path.append(os.path.join(os.getcwd(), '..', 'src'))\n
from src.preprocess import load_data, preprocess_data, aggregate_daily_sales

In [ ]:
df = load_data('../data/superstore.csv')\n
df = preprocess_data(df)\n
daily = aggregate_daily_sales(df)\n
daily.to_csv('../outputs/daily_sales.csv', index=False)\n
print(daily.head())

In [ ]:
fig, ax = plt.subplots(figsize=(12,4))\n
daily.set_index('date')['sales'].plot(ax=ax)\n
plt.savefig('../outputs/forecast_plot.png')\n
plt.show()\n
\n
print(daily.describe())

In [ ]:
daily = daily.set_index('date')\n
daily['lag1'] = daily['sales'].shift(1)\n
daily['month'] = daily.index.month\n
daily_ml = daily.dropna()\n
X = daily_ml.drop('sales', 1)\n
y = daily_ml['sales']\n
split = int(0.8*len(X))\n
Xtr, Xte = X[:split], X[split:]\n
ytr, yte = y[:split], y[split:]

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)\n
rf.fit(Xtr, ytr)\n
pred = rf.predict(Xte)\n
mae = mean_absolute_error(yte, pred)\n
rmse = np.sqrt(mean_squared_error(yte, pred))\n
joblib.dump(rf, '../models/rf_model.pkl')\n
print(f'MAE: {mae:.2f}, RMSE: {rmse:.2f}')

In [ ]:
arima = ARIMA(ytr, order=(5,1,0))\n
fit = arima.fit()\n
ar_pred = fit.forecast(steps=len(yte))\n
ar_mae = mean_absolute_error(yte, ar_pred)\n
ar_rmse = np.sqrt(mean_squared_error(yte, ar_pred))\n
joblib.dump(fit, '../models/arima_model.pkl')\n
print(f'ARIMA MAE: {ar_mae:.2f}, RMSE: {ar_rmse:.2f}')

In [ ]:
dates = pd.date_range(daily.index[-1] + pd.Timedelta(1, 'D'), periods=30)\n
lastX = X.iloc[[-1]]\n
fc = []\n
cx = lastX.copy()\n
for _ in range(30):\n
    p = rf.predict(cx)[0]\n
    fc.append(p)\n
    cx['lag1'] = p\n
fc_df = pd.DataFrame({'Date': dates, 'RF_Forecast': fc})\n
fc_df.to_csv('../outputs/forecast_30_days.csv', index=False)\n
\n
metrics = f'RF MAE: {mae:.2f}\nARIMA MAE: {ar_mae:.2f}'\n
with open('../outputs/metrics.txt', 'w') as f: f.write(metrics)